# 🛒 42 Workshop: Evaluation Pipeline

Welcome to the workshop! In this notebook, we will go through the evaluation pipeline step by step.

We will:
1. Configure the environment.
2. Load the agent and evaluation data.
3. Upload the dataset to Langfuse.
4. Run an experiment to measure the agent's performance.
5. Create a human annotation queue in Langfuse.

### 1. Preparation & Imports

First, we load the necessary libraries and configure the environment. Make sure you have a `.env` file with the appropriate Langfuse and Google Cloud credentials.

In [1]:
import os
import nest_asyncio
from dotenv import load_dotenv
from loguru import logger

# Allows nested event loops in Jupyter
nest_asyncio.apply()

# Load environment variables
load_dotenv()

True

### 2. Load Evaluation Data

We read the prepared test cases from `data/evaluation_data/eval_data.json`.

In [3]:
from app.models.evaluation import EvalSuite
from app.services.database import DATA_DIR

def read_evaluation_data():
    path = DATA_DIR / "evaluation_data" / "eval_data.json"
    print(f"Loading data from: {path}")
    json_string = path.read_text()
    return EvalSuite.model_validate_json(json_string)

eval_data = read_evaluation_data()
print(f"Number of test cases: {len(eval_data.items)}")
print("Example:", eval_data.items[0].question)

Loading data from: /Users/malich/repos/42-Agent-workshop/data/evaluation_data/eval_data.json
Number of test cases: 5
Example: What is the best-selling item?


### 3. Upload Dataset to Langfuse

We use the `langfuse_client` to create or update a dataset in Langfuse.

In [4]:
from app.services.langfuse_client import langfuse_client as langfuse
from langfuse.api import NotFoundError

DATASET_NAME = "42_workshop_notebook_eval"

def upload_evaluation_data(evaluation_data: EvalSuite):
    try:
        langfuse.get_dataset(DATASET_NAME)
        print(f"Dataset '{DATASET_NAME}' already exists.")
    except Exception:
        print(f"Creating dataset '{DATASET_NAME}'...")
        langfuse.create_dataset(name=DATASET_NAME)

    for item in evaluation_data.items:
        langfuse.create_dataset_item(
            dataset_name=DATASET_NAME,
            input={"question": item.question},
            expected_output=item.expected_output,
            metadata=item.metadata.model_dump() if item.metadata else {},
        )
    print("Upload complete.")

upload_evaluation_data(eval_data)

2026-04-13 00:30:49.766 | INFO     | app.services.langfuse_client:__new__:12 - Langfuse client is authenticated and ready!


Creating dataset '42_workshop_notebook_eval'...
Upload complete.


## 4. Create Evaluators

In [6]:
from deepeval.metrics import ToolCorrectnessMetric, AnswerRelevancyMetric
from deepeval.test_case import LLMTestCase, ToolCall
from langfuse import Evaluation
from app.evaluation.providers import VertexGemini

## LLM-based evaluation: Answer Relevancy
async def answer_relevancy(
    input: dict,
    output: dict,
    expected_output: str,
    metadata: dict | None,
    **kwargs,
):
    metric = AnswerRelevancyMetric(
        threshold=0.7, model=VertexGemini(), include_reason=True
    )

    test_case = LLMTestCase(
        input=input["question"],
        actual_output=output["output"],
        expected_output=expected_output,
    )

    res = await metric.a_measure(test_case)

    return Evaluation(name="answer_relevancy", value=res, comment="LLM-based evaluation of response relevance.")


## Trajectory-based evaluation: Tool Calling Accuracy
async def tool_calling_accuracy(
    input: dict,
    output: dict,
    expected_output: str,
    metadata: dict | None,
    **kwargs,
):
    def convert_to_tool_calls(calls: list[str]) -> list[ToolCall]:
        return [ToolCall(name=call, input_parameters=None) for call in calls]

    # Convert the actual tool calls captured from the agent run
    tool_calls_list = [
        ToolCall(name=call.tool_name, input_parameters=None)
        for call in output["eval_metadata"]["tool_calls"]
    ]

    metric = ToolCorrectnessMetric(model=VertexGemini())

    test_case = LLMTestCase(
        input=input["question"],
        actual_output=output["output"],
        expected_output=expected_output,
        tools_called=tool_calls_list,
        expected_tools=convert_to_tool_calls(metadata["expected_tool_calls"] if metadata else []),
    )

    res = await metric.a_measure(test_case)
    return Evaluation(name="tool_call_accuracy", value=res)

### 5. Run Experiment

Now we let the agent run against the dataset. We measure `answer_relevancy` and `tool_calling_accuracy`.

In [7]:
from app.services.agent import retail_agent
from app.evaluation.utils import extract_tool_calls

async def call_agent(*, item, **kwargs):
    # We use the Retail Agent from the app
    run_results = await retail_agent.run(item.input["question"])
    
    return {
        "output": run_results.output.message,
        "eval_metadata": {"tool_calls": extract_tool_calls(run_results)},
    }

print("Starting experiment...")
eval_dataset = langfuse.get_dataset(DATASET_NAME)
evaluation_result = eval_dataset.run_experiment(
    name="Workshop Notebook Run",
    task=call_agent,
    evaluators=[answer_relevancy, tool_calling_accuracy],
)
print("Experiment finished.")

Starting experiment...


/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/pydantic_ai/profiles/google.py:41: UserWarning: `additionalProperties` is not supported by Gemini; it will be removed from the tool JSON schema. Full schema: {'properties': {'message': {'description': 'Your conversational response to the user.', 'type': 'string'}, 'best_seller_identified': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'description': 'Name of the best seller, if asked.'}, 'restock_orders_placed': {'anyOf': [{'additionalProperties': {'type': 'integer'}, 'type': 'object'}, {'type': 'null'}], 'description': 'Map of product names to quantities ordered, if any.'}}, 'required': ['message', 'best_seller_identified', 'restock_orders_placed'], 'title': 'AssistantResponse', 'type': 'object'}

Source of additionalProperties within the full schema: {'type': 'object', 'additionalProperties': {'type': 'integer'}}

If this came from a field with a type like `dict[str, MyType]`, that field will always be empty

/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/vertexai/generative_models/_generative_mod
els.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For 
details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()

/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

...

/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/rich/live.py:260: UserWarning: install 
"ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/Users/malich/repos/42-Agent-workshop/.venv/lib/python3.13/site-packages/vertexai/generative_models/_generative_models.py:433: UserWarning: This feature is deprecated as of June 24, 2025 and will be removed on June 24, 2026. For details, see https://cloud.google.com/vertex-ai/generative-ai/docs/deprecations/genai-vertexai-sdk.
  warning_logs.show_deprecation_warning()


Experiment finished.


### 5. Human-in-the-Loop: Annotation Queue

To manually review the results, we create a queue in Langfuse.

In [9]:
    ### Optional - Get all score configuation IDs
    # Fetch a paginated list of all score configs
    configs = langfuse.api.score_configs.get()

    # Loop through and print names and their corresponding IDs
    for config in configs.data:
        print(f"Name: {config.name} | ID: {config.id}")

Name: Human Answer Relevancy | ID: cmnvjr511000aqf06j49s7ixw
Name: Main Driver Recall | ID: cmn2vtwdu0001pg07bzhj04pb


In [10]:
def create_annotation_queue(eval_results):
    queue_name = "workshop_queue_" + eval_results.run_name
    print(f"Creating queue: {queue_name}")
    
    # Note: score_config_ids depend on your Langfuse instance
    queue = langfuse.api.annotation_queues.create_queue(
        name=queue_name, score_config_ids=["cmnvjr511000aqf06j49s7ixw"]
    )

    for item in eval_results.item_results:
        trace = langfuse.api.trace.get(trace_id=item.trace_id)
        # We annotate the first observation (usually the LLM call)
        langfuse.api.annotation_queues.create_queue_item(
            queue_id=queue.id,
            object_id=trace.observations[0].id,
            object_type="OBSERVATION",
        )
    print(f"Queue created. ID: {queue.id}")

create_annotation_queue(evaluation_result)

Creating queue: workshop_queue_Workshop Notebook Run - 2026-04-12T22:36:53.923360Z
Queue created. ID: cmnwchhic0096qf06mddunyla


# 🛠️ Workshop Exercises

Congratulations! You have successfully executed the base pipeline. Now it's time to dive deeper. Complete the following tasks to improve the evaluation.

---

### Task 1: Implement Your Own Metric

In the retail world, it's important that the agent remains friendly and professional. Your task is to implement a new metric that evaluates the agent's **Tone/Professionalism**.

Use the `deepeval.metrics.GEval` class for this.

**Steps:**
1. Define a `GEval` metric.
2. Give it a name (e.g., "Professionalism").
3. Define the `evaluation_params` (we need `actual_output`).
4. Write the `evaluation_steps` (What should the LLM look for? Politeness, grammar, helpfulness).
5. Integrate the metric into a new `run_experiment` call.

In [ ]:
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCaseParams, LLMTestCase
from app.evaluation.providers import VertexGemini
from langfuse import Evaluation

# TODO: Implement the Professionalism metric
# professional_metric = GEval(
#     name="Professionalism",
#     model=VertexGemini(),
#     evaluation_params=[LLMTestCaseParams.ACTUAL_OUTPUT],
#     evaluation_steps=[
#         "Check if the agent is polite and helpful.",
#         "Check if the tone is professional.",
#         "Ensure no slang or offensive language is used."
#     ]
# )

async def professional_tone(input, output, expected_output, metadata, **kwargs):
    # TODO: Implement the wrapper for Langfuse
    # test_case = LLMTestCase(input=input["question"], actual_output=output["output"])
    # res = await professional_metric.a_measure(test_case)
    # return Evaluation(name="professionalism", value=res)
    pass

# Run the experiment again with the new metric
# evaluation_result_v2 = eval_dataset.run_experiment(
#     name="Workshop Exercise Run",
#     task=call_agent,
#     evaluators=[answer_relevancy, tool_calling_accuracy, professional_tone],
# )

### Task 2: Expand the Dataset

Good evaluations need good data. Think of a new test case for the agent. This should trigger a tool interaction (e.g., price change or restock).

**Steps:**
1. Create a new `EvalItem` (Input, Expected Output, Metadata with `expected_tool_calls`).
2. Use the `langfuse_client.create_dataset_item` call to manually add this case to the dataset.
3. Run the experiment again and check the new trace in Langfuse.

In [ ]:
# TODO: Create a new test item
# new_input = "Set the price of laptop to 1200$"
# new_expected = "I've updated the price of the laptop to $1200."
# new_metadata = {"expected_tool_calls": ["adjust_price"]}

# langfuse.create_dataset_item(
#     dataset_name=DATASET_NAME,
#     input={"question": new_input},
#     expected_output=new_expected,
#     metadata=new_metadata,
# )

# Run the experiment again and check Langfuse!
# eval_dataset.run_experiment(...)